# Kafka End-to-End Ingestion Examples

This notebook walks through a fuller Kafka ingestion flow in Databricks using bronze, silver, and gold layers.

The flow covers:

- reading raw messages from Kafka
- landing raw events into bronze with Kafka metadata
- validating and deduplicating events in silver
- publishing a gold aggregate for downstream consumers

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

In [ ]:
kafka_bootstrap_servers = "broker1:9092,broker2:9092"
kafka_topic = "orders.events"
bronze_checkpoint_path = "/Volumes/main/demo/checkpoints/kafka_orders_bronze"
silver_checkpoint_path = "/Volumes/main/demo/checkpoints/kafka_orders_silver"
gold_checkpoint_path = "/Volumes/main/demo/checkpoints/kafka_orders_gold"
bronze_table = "main.demo.orders_kafka_bronze"
silver_table = "main.demo.orders_kafka_silver"
gold_table = "main.demo.orders_kafka_gold_daily"

orders_event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_region", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("amount", DoubleType(), True)
])

## Bronze stream

Bronze keeps the raw message and Kafka offsets so the ingestion can be replayed or audited later.

In [ ]:
bronze_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "latest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_ts"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_payload")
    )
    .withColumn("payload", F.from_json("raw_payload", orders_event_schema))
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_ts",
        "message_key",
        "raw_payload",
        F.col("payload.*"),
        F.current_timestamp().alias("bronze_ingest_ts")
    )
)

bronze_stream_df

In [ ]:
bronze_query = (
    bronze_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

## Silver validation and deduplication

Silver keeps only valid event records and removes duplicates by event identifier.

In [ ]:
silver_stream_df = (
    spark.readStream.table(bronze_table)
    .withColumn("event_ts", F.to_timestamp("event_ts"))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("customer_region", F.upper(F.trim("customer_region")))
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("event_ts").isNotNull())
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_id"])
)

silver_stream_df

In [ ]:
silver_query = (
    silver_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

## Gold aggregate

Gold publishes a business-ready daily revenue summary by region.

In [ ]:
gold_stream_df = (
    spark.readStream.table(silver_table)
    .filter(F.col("event_type") == "order_completed")
    .withColumn("order_date", F.to_date("event_ts"))
    .groupBy("order_date", "customer_region")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("amount").alias("gross_sales")
    )
)

gold_stream_df

In [ ]:
gold_query = (
    gold_stream_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", gold_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(gold_table)
)

In [ ]:
display(spark.table(gold_table).orderBy(F.col("order_date").desc(), "customer_region"))

## Production notes

- Keep the raw Kafka payload in bronze for replay and audit.
- Use watermarking and deduplication in silver to handle retries and late events.
- Publish small, stable aggregates in gold for downstream analytics.
- Monitor consumer lag, checkpoint health, and schema evolution for production Kafka pipelines.